#### 1) Build the 1000-case CSVs in Notes_for_1000_cases

In [4]:
import os
import pandas as pd

DERIVED = "/nfs/turbo/umms-atjanke/liuwent/Notes_Summarize_Generation/InputData/Notes/derived_tables"
OUTDIR = "/nfs/turbo/umms-atjanke/liuwent/Notes_Feature_Abstraction/Notes_for_1000_cases"

os.makedirs(OUTDIR, exist_ok=True)

enc = pd.read_parquet(f"{DERIVED}/encounter_master.parquet")
notes = pd.read_parquet(f"{DERIVED}/notes_raw.parquet")

eligible = enc[
    (enc.pe_suspected == 1) &
    (enc.in_chest_pain_sob_cohort == 1) &
    (enc.has_triage_note == 1) &
    (enc.has_provider_note == 1) &
    (enc.has_ct_report == 1) &
    (enc.included_in_labeled_200.fillna(0) == 0)
].copy()

if len(eligible) < 1000:
    raise ValueError(f"Only {len(eligible)} eligible encounters found, cannot sample 1000.")

sample_1000 = eligible.sample(n=1000, random_state=42).copy()
sample_ids = set(sample_1000["EncounterCsn"].tolist())

sample_1000[["EncounterCsn"]].to_csv(
    f"{OUTDIR}/pe_workup_sample1000_encounters.csv",
    index=False
)

SEP = "\n\n" + ("=" * 80) + "\n\n"

def build_note_block(row):
    parts = [f"SOURCE={row['note_source']}"]
    if pd.notna(row.get("note_subtype")):
        parts.append(f"SUBTYPE={row['note_subtype']}")
    if pd.notna(row.get("note_time")):
        parts.append(f"TIME={row['note_time']}")
    if pd.notna(row.get("note_id")):
        parts.append(f"NOTE_ID={row['note_id']}")
    header = " | ".join(parts)
    text = "" if pd.isna(row["text"]) else str(row["text"]).strip()
    return f"[{header}]\n{text}"

# -------------------------
# Task 1: triage + provider
# force triage first, then provider
# -------------------------
tp = notes[
    notes["EncounterCsn"].isin(sample_ids) &
    notes["note_source"].isin(["triage", "provider"])
].copy()

source_order = {
    "triage": 0,
    "provider": 1,
}
tp["source_sort"] = tp["note_source"].map(source_order).fillna(9)

tp = tp.sort_values(
    ["EncounterCsn", "source_sort", "note_time", "note_order_within_encounter"],
    na_position="last"
)

tp["block"] = tp.apply(build_note_block, axis=1)

tp_agg = (
    tp.groupby("EncounterCsn", dropna=False)
      .agg(
          Note_Count=("note_id", "nunique"),
          Text=("block", lambda s: SEP.join([x for x in s if str(x).strip() != ""]))
      )
      .reset_index()
)

tp_agg.to_csv(
    f"{OUTDIR}/notes-for-1000-cases.csv",
    index=False
)

# -------------------------
# Task 2: CT reports only
# -------------------------
ct = notes[
    notes["EncounterCsn"].isin(sample_ids) &
    (notes["note_source"] == "ct_report")
].copy()

ct = ct.sort_values(
    ["EncounterCsn", "note_time", "note_order_within_encounter"],
    na_position="last"
)

ct["block"] = ct.apply(build_note_block, axis=1)

ct_agg = (
    ct.groupby("EncounterCsn", dropna=False)
      .agg(
          Note_Count=("note_id", "nunique"),
          Text=("block", lambda s: SEP.join([x for x in s if str(x).strip() != ""]))
      )
      .reset_index()
)

ct_agg.to_csv(
    f"{OUTDIR}/ct-reports-for-1000-cases.csv",
    index=False
)

print("Eligible encounters:", len(eligible))
print("Sampled encounters:", len(sample_1000))
print("notes-for-1000-cases.csv:", tp_agg.shape)
print("ct-reports-for-1000-cases.csv:", ct_agg.shape)

Eligible encounters: 6800
Sampled encounters: 1000
notes-for-1000-cases.csv: (1000, 3)
ct-reports-for-1000-cases.csv: (1000, 3)


In [5]:
import pandas as pd

df = pd.read_csv("/nfs/turbo/umms-atjanke/liuwent/Notes_Feature_Abstraction/Notes_for_1000_cases/notes-for-1000-cases.csv")
print(df.loc[0, "Text"][:2000])

[SOURCE=triage | SUBTYPE=ED Triage Note | TIME=2023-01-02 18:26:00 | NOTE_ID=136281853]
HPI: pt was admitted and discharged christmas day with pneumonia. Discharged with medication but symptoms of cough and SOB are worsening. Cancer pt and hx of transplants. Objective: no distress. Breathing unlabored. Skin pwd. Alert and oriented


[SOURCE=provider | SUBTYPE=ED Provider Notes | TIME=2023-01-02 18:16:00 | NOTE_ID=136309828]
UNIVERSITY OF MICHIGAN Adult Emergency Services Evaluation Note Mark Smyth is a 60 y.o. male who presented to the Emergency Department at 1816 on 1/2/23. HISTORY Chief Complaint and History of Present Illness Patient with history of ESRD due to hepatorenal syndrome/ATN s/p liver transplant in 2019 and renal transplant in 2021, stage IV metastatic NSCLC diagnosed 05/2022 s/p 4 cycles (carbo/taxol on 9/10/22) and palliative radiation of spinal met (07/2022), HTN, HLD, anxiety, and GERD who presents to the ED with dyspnea on exertion and fatigue. He was recently admitt

#### 2) Generate updated shell scripts from the Excel schemas

In [6]:
import os
import pandas as pd

OUTDIR = "/nfs/turbo/umms-atjanke/liuwent/Notes_Feature_Abstraction/Notes_for_1000_cases"
PE_SCHEMA_XLSX = "/nfs/turbo/umms-atjanke/liuwent/Schema/pe-schema.xlsx"
CT_SCHEMA_XLSX = "/nfs/turbo/umms-atjanke/liuwent/Schema/ct-chest-schema.xlsx"

os.makedirs(OUTDIR, exist_ok=True)


def find_col(df, candidates):
    lower_map = {c.strip().lower(): c for c in df.columns}
    for cand in candidates:
        key = cand.strip().lower()
        if key in lower_map:
            return lower_map[key]
    raise KeyError(f"Could not find any of these columns: {candidates}. Found: {list(df.columns)}")


def load_schema_vars(xlsx_path):
    df = pd.read_excel(xlsx_path).copy()

    name_col = find_col(df, ["Variable Name", "Variable", "Name"])
    type_col = find_col(df, ["Type", "Kind"])
    instr_col = find_col(df, ["Instructions", "Instruction", "Description", "Prompt"])

    vars_out = []
    for _, row in df.iterrows():
        name = str(row[name_col]).strip()
        kind = str(row[type_col]).strip().lower()
        desc = str(row[instr_col]).strip()

        if not name or name.lower() == "nan":
            continue
        if not desc or desc.lower() == "nan":
            desc = f"Extract {name} from the note."

        if kind in {"yes/no", "bool", "boolean"}:
            kind = "yn"
        elif kind in {"yes/no/uncertain", "ynu"}:
            kind = "ynu"
        elif kind in {"presence/absence", "presence_absence", "presence"}:
            kind = "presence"
        elif kind not in {"yn", "ynu", "presence", "text", "text_opt"}:
            kind = "presence"

        vars_out.append(f"{name}:{kind}:{desc}")
    return vars_out


def write_gpt_script(
    script_path,
    job_name,
    model,
    input_csv,
    output_name,
    json_name,
    var_specs,
):
    with open(script_path, "w", encoding="utf-8") as f:
        f.write(f"""#!/bin/bash
# Auto-generated GPT script for 1000-case run

#SBATCH --job-name={job_name}
#SBATCH --account=atjanke0
#SBATCH --partition=standard
#SBATCH --time=24:00:00
#SBATCH --nodes=1
#SBATCH --cpus-per-task=2
#SBATCH --mem=64G
#SBATCH --output=./%x-%j
#SBATCH --error=./%x-%j
#SBATCH --mail-user=liuwent@med.umich.edu
#SBATCH --mail-type=BEGIN,END,FAIL

set -e -o pipefail

if [[ ${{SLURM_JOB_NODELIST:-}} ]]; then
  echo "Running on:"
  scontrol show hostnames "$SLURM_JOB_NODELIST"
fi

source "$HOME/.bashrc"
conda activate PE

WORKDIR="{OUTDIR}"
cd "$WORKDIR" || exit 1

set -a
source "/nfs/turbo/umms-atjanke/liuwent/gpt.env"
set +a
export HTTPS_PROXY="http://proxy1.arc-ts.umich.edu:3128/"
export HTTP_PROXY="$HTTPS_PROXY"
export NO_PROXY=""
export no_proxy=""
export OPENAI_BASE_URL="${{OPENAI_BASE_URL:-$OPENAI_API_BASE}}"

mkdir -p outputs

python -u ./batch-abstract-notes-logged.py \\
  --input "./{input_csv}" \\
  --output "{output_name}" \\
  --note-col Text \\
  --id-col EncounterCsn \\
  --script ./llm-chart-abstraction-call.py \\
""")
        for spec in var_specs:
            escaped = spec.replace('"', '\\"')
            f.write(f'  --var "{escaped}" \\\n')

        f.write(f"""  --exp-all \\
  --quote-per-var \\
  --repair \\
  --rps 2 \\
  --checkpoint-every 10 \\
  --model {model} \\
  --json-out "{json_name}"
""")


def write_mistral7b_script(
    script_path,
    job_name,
    input_csv,
    output_prefix,
    var_specs,
    num_shards=8,
):
    with open(script_path, "w", encoding="utf-8") as f:
        f.write(f"""#!/bin/bash

#SBATCH --job-name={job_name}
#SBATCH --account=atjanke0
#SBATCH --partition=gpu-rtx6000
#SBATCH --qos=normal
#SBATCH --nodes=1
#SBATCH --ntasks=1
#SBATCH --cpus-per-task=8
#SBATCH --mem=64G
#SBATCH --gres=gpu:1
#SBATCH --array=0-{num_shards-1}%4
#SBATCH --time=12:00:00
#SBATCH --output=./%x-%A_%a.out
#SBATCH --error=./%x-%A_%a.err
#SBATCH --mail-user=liuwent@umich.edu
#SBATCH --mail-type=BEGIN,END,FAIL

set -euo pipefail

module purge
module load cuda/12.8.1
module load gcc/10.3.0

set +u
source "$HOME/.bashrc"
set -u
conda activate PE
export PYTHONNOUSERSITE=1

WORKDIR="{OUTDIR}"
cd "$WORKDIR"
mkdir -p outputs

export LLM_API_PROVIDER="mistral_local"
export MISTRAL_MODEL_DIR="/nfs/turbo/umms-atjanke/liuwent/Notes_Feature_Abstraction/Mistral/7B"

export MISTRAL_USE_4BIT=0
export MISTRAL_MAX_NEW_TOKENS=1200
export MISTRAL_TEMPERATURE=0.0
export MISTRAL_USE_CACHE=0
export MISTRAL_CHUNK_TOKENS=9000
export MISTRAL_MAX_CHUNKS=0
export PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True"

echo "=== ENV ==="
hostname
echo "WORKDIR=$WORKDIR"
echo "SLURM_ARRAY_TASK_ID=${{SLURM_ARRAY_TASK_ID}}"
nvidia-smi

if [[ "${{SLURM_ARRAY_TASK_ID}}" == "0" ]]; then
  echo "=== SINGLE NOTE SMOKE TEST (shard 0 only) ==="
  python - << 'PY'
import pandas as pd, subprocess, sys
df = pd.read_csv("./{input_csv}")
note = str(df["Text"].iloc[0])
cmd = [
    sys.executable, "./llm-chart-abstraction-call_Mistral7B.py",
    "--api-provider", "mistral_local",
    "--var", "smoke_test_field:presence:Does the note mention a clinically relevant finding?",
    "--repair"
]
p = subprocess.run(cmd, input=note, text=True, capture_output=True)
print("returncode:", p.returncode)
print("STDERR tail:", (p.stderr or "")[-800:])
print("STDOUT head:", (p.stdout or "")[:500])
PY
fi

NUM_SHARDS={num_shards}
SHARD=${{SLURM_ARRAY_TASK_ID}}

python -u ./batch-abstract-notes-logged_Mistral7B.py \\
  --input ./{input_csv} \\
  --output {output_prefix}_shard${{SHARD}}.parquet \\
  --note-col Text \\
  --id-col EncounterCsn \\
  --script ./llm-chart-abstraction-call_Mistral7B.py \\
  --model mistral-7b \\
  --api-provider mistral_local \\
  --num-shards ${{NUM_SHARDS}} \\
  --shard-index ${{SHARD}} \\
  --repair \\
  --exp-all \\
  --quote-per-var \\
  --rpm 100000 \\
  --checkpoint-every 1 \\
  --json-out "debug-{output_prefix}-job${{SLURM_JOB_ID}}-shard${{SHARD}}.json" \\
""")
        for spec in var_specs:
            escaped = spec.replace('"', '\\"')
            f.write(f'  --var "{escaped}" \\\n')


def write_mistral70b_script(
    script_path,
    job_name,
    input_csv,
    output_prefix,
    var_specs,
    num_shards=8,
):
    with open(script_path, "w", encoding="utf-8") as f:
        f.write(f"""#!/bin/bash

#SBATCH --job-name={job_name}
#SBATCH --account=atjanke0
#SBATCH --partition=gpu-rtx6000
#SBATCH --qos=normal
#SBATCH --nodes=1
#SBATCH --ntasks=1
#SBATCH --cpus-per-task=16
#SBATCH --mem=192G
#SBATCH --gres=gpu:1
#SBATCH --array=0-{num_shards-1}%2
#SBATCH --time=12:00:00
#SBATCH --output=./%x-%A_%a.out
#SBATCH --error=./%x-%A_%a.err
#SBATCH --mail-user=liuwent@umich.edu
#SBATCH --mail-type=BEGIN,END,FAIL

set -euo pipefail

module purge
module load cuda/12.8.1
module load gcc/10.3.0

set +u
source "$HOME/.bashrc"
set -u
conda activate PE
export PYTHONNOUSERSITE=1

WORKDIR="{OUTDIR}"
cd "$WORKDIR"
mkdir -p outputs

export LLM_API_PROVIDER="hf_local"
export HF_MODEL_ID="cookinai/OrcaHermes-Mistral-70B-miqu"
export HF_USE_4BIT=1
export HF_DTYPE="bfloat16"
export HF_MAX_NEW_TOKENS=1200
export HF_TEMPERATURE=0.0
export HF_USE_CACHE=0
export HF_CHUNK_TOKENS=3500
export HF_MAX_CHUNKS=0
export PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True"
export TOKENIZERS_PARALLELISM=false
export HF_HOME="${{WORKDIR}}/.hf_cache"
export TRANSFORMERS_CACHE="${{HF_HOME}}/transformers"

echo "=== ENV ==="
hostname
echo "WORKDIR=$WORKDIR"
echo "SLURM_ARRAY_TASK_ID=${{SLURM_ARRAY_TASK_ID}}"
nvidia-smi

if [[ "${{SLURM_ARRAY_TASK_ID}}" == "0" ]]; then
  echo "=== SINGLE NOTE SMOKE TEST (shard 0 only) ==="
  python - << 'PY'
import pandas as pd, subprocess, sys
df = pd.read_csv("./{input_csv}")
note = str(df["Text"].iloc[0])
cmd = [
    sys.executable, "./llm-chart-abstraction-call_Mistral70B.py",
    "--api-provider", "hf_local",
    "--model", "cookinai/OrcaHermes-Mistral-70B-miqu",
    "--repair",
    "--quote-per-var",
    "--var", "smoke_test_field:presence:Does the note mention a clinically relevant finding?"
]
p = subprocess.run(cmd, input=note, text=True, capture_output=True)
print("returncode:", p.returncode)
print("STDERR tail:", (p.stderr or "")[-1200:])
print("STDOUT head:", (p.stdout or "")[:1200])
PY
fi

NUM_SHARDS={num_shards}
SHARD=${{SLURM_ARRAY_TASK_ID}}

python -u ./batch-abstract-notes-logged_Mistral70B.py \\
  --input ./{input_csv} \\
  --output {output_prefix}_shard${{SHARD}}.parquet \\
  --note-col Text \\
  --id-col EncounterCsn \\
  --script ./llm-chart-abstraction-call_Mistral70B.py \\
  --model cookinai/OrcaHermes-Mistral-70B-miqu \\
  --api-provider hf_local \\
  --num-shards ${{NUM_SHARDS}} \\
  --shard-index ${{SHARD}} \\
  --repair \\
  --quote-per-var \\
  --exp-all \\
  --rpm 100000 \\
  --checkpoint-every 1 \\
  --json-out "debug-{output_prefix}-job${{SLURM_JOB_ID}}-shard${{SHARD}}.json" \\
""")
        for spec in var_specs:
            escaped = spec.replace('"', '\\"')
            f.write(f'  --var "{escaped}" \\\n')


def write_merge_script(script_path, prefix, merged_name, expected_shards=8):
    with open(script_path, "w", encoding="utf-8") as f:
        f.write(f"""#!/bin/bash
set -euo pipefail

cd "{OUTDIR}"

python - <<'PY'
import glob
import pandas as pd

EXPECTED_SHARDS = {expected_shards}
files = sorted(glob.glob("./outputs/{prefix}_shard*.parquet"))
print("Found", len(files), "parquets")

if len(files) != EXPECTED_SHARDS:
    raise ValueError(f"Expected {{EXPECTED_SHARDS}} shard parquet files, found {{len(files)}}")

df = pd.concat([pd.read_parquet(f) for f in files], ignore_index=True)
df = df.sort_values("EncounterCsn")
df.to_parquet("./outputs/{merged_name}", index=False)
print("Wrote ./outputs/{merged_name} with rows:", len(df))
PY

python - <<'PY'
import glob
import json

files = sorted(glob.glob("./outputs/{prefix}_shard*.log.*.json"))
print("Found", len(files), "shard log JSONs")
if files:
    merged = []
    for fp in files:
        with open(fp, "r", encoding="utf-8") as f:
            merged.append(json.load(f))
    out_path = "./outputs/{prefix}_ALL_shard_runlogs.json"
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(merged, f, indent=2)
    print("Wrote:", out_path)
else:
    print("No shard log JSON files found.")
PY

python - <<'PY'
import glob
import json
from collections import OrderedDict

files = sorted(glob.glob("./outputs/debug-{prefix}-job*-shard*.json"))
print("Found", len(files), "debug json files for prefix: {prefix}")

all_rows = []
for fp in files:
    try:
        with open(fp, "r", encoding="utf-8") as f:
            data = json.load(f)
    except Exception as e:
        print("Skipping unreadable:", fp, "error:", e)
        continue

    if isinstance(data, list):
        all_rows.extend(data)
    elif isinstance(data, dict):
        all_rows.append(data)

dedup = OrderedDict()
no_id = []
for r in all_rows:
    if isinstance(r, dict) and "_id" in r and r["_id"] is not None:
        dedup[str(r["_id"])] = r
    else:
        no_id.append(r)

merged = list(dedup.values()) + no_id
out_path = "./outputs/{prefix}_ALL_debug_merged.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(merged, f, indent=2, ensure_ascii=False)

print("Merged debug records:", len(merged))
print("Wrote:", out_path)
PY
""")


pe_vars = load_schema_vars(PE_SCHEMA_XLSX)
ct_vars = load_schema_vars(CT_SCHEMA_XLSX)

# GPT scripts
write_gpt_script(
    f"{OUTDIR}/PE_GPT5.sh",
    "PE_GPT5_1000",
    "gpt-5",
    "notes-for-1000-cases.csv",
    "notes-for-1000-cases-pe-schema-gpt5.parquet",
    "notes-for-1000-cases-pe-schema-gpt5.json",
    pe_vars,
)
write_gpt_script(
    f"{OUTDIR}/PE_GPT5_Mini.sh",
    "PE_GPT5Mini_1000",
    "gpt-5-mini",
    "notes-for-1000-cases.csv",
    "notes-for-1000-cases-pe-schema-gpt5mini.parquet",
    "notes-for-1000-cases-pe-schema-gpt5mini.json",
    pe_vars,
)
write_gpt_script(
    f"{OUTDIR}/PE_GPT5_Nano.sh",
    "PE_GPT5Nano_1000",
    "gpt-5-nano",
    "notes-for-1000-cases.csv",
    "notes-for-1000-cases-pe-schema-gpt5nano.parquet",
    "notes-for-1000-cases-pe-schema-gpt5nano.json",
    pe_vars,
)

write_gpt_script(
    f"{OUTDIR}/CT_GPT5.sh",
    "CT_GPT5_1000",
    "gpt-5",
    "ct-reports-for-1000-cases.csv",
    "ct-reports-for-1000-cases-ct-schema-gpt5.parquet",
    "ct-reports-for-1000-cases-ct-schema-gpt5.json",
    ct_vars,
)
write_gpt_script(
    f"{OUTDIR}/CT_GPT5_Mini.sh",
    "CT_GPT5Mini_1000",
    "gpt-5-mini",
    "ct-reports-for-1000-cases.csv",
    "ct-reports-for-1000-cases-ct-schema-gpt5mini.parquet",
    "ct-reports-for-1000-cases-ct-schema-gpt5mini.json",
    ct_vars,
)
write_gpt_script(
    f"{OUTDIR}/CT_GPT5_Nano.sh",
    "CT_GPT5Nano_1000",
    "gpt-5-nano",
    "ct-reports-for-1000-cases.csv",
    "ct-reports-for-1000-cases-ct-schema-gpt5nano.parquet",
    "ct-reports-for-1000-cases-ct-schema-gpt5nano.json",
    ct_vars,
)

# Mistral 7B scripts
write_mistral7b_script(
    f"{OUTDIR}/PE_Mistral7B.sh",
    "PE_Mistral7B_1000",
    "notes-for-1000-cases.csv",
    "notes-for-1000-cases-pe-schema-mistral7b",
    pe_vars,
)
write_mistral7b_script(
    f"{OUTDIR}/CT_Mistral7B.sh",
    "CT_Mistral7B_1000",
    "ct-reports-for-1000-cases.csv",
    "ct-reports-for-1000-cases-ct-schema-mistral7b",
    ct_vars,
)

# Mistral 70B scripts
write_mistral70b_script(
    f"{OUTDIR}/PE_Mistral70B.sh",
    "PE_Mistral70B_1000",
    "notes-for-1000-cases.csv",
    "notes-for-1000-cases-pe-schema-mistral70b",
    pe_vars,
)
write_mistral70b_script(
    f"{OUTDIR}/CT_Mistral70B.sh",
    "CT_Mistral70B_1000",
    "ct-reports-for-1000-cases.csv",
    "ct-reports-for-1000-cases-ct-schema-mistral70b",
    ct_vars,
)

# Merge scripts
write_merge_script(
    f"{OUTDIR}/PE_Mistral7B_Merge.sh",
    "notes-for-1000-cases-pe-schema-mistral7b",
    "ALL_notes-for-1000-cases-pe-schema-mistral7b.parquet",
    expected_shards=8,
)
write_merge_script(
    f"{OUTDIR}/CT_Mistral7B_Merge.sh",
    "ct-reports-for-1000-cases-ct-schema-mistral7b",
    "ALL_ct-reports-for-1000-cases-ct-schema-mistral7b.parquet",
    expected_shards=8,
)
write_merge_script(
    f"{OUTDIR}/PE_Mistral70B_Merge.sh",
    "notes-for-1000-cases-pe-schema-mistral70b",
    "ALL_notes-for-1000-cases-pe-schema-mistral70b.parquet",
    expected_shards=8,
)
write_merge_script(
    f"{OUTDIR}/CT_Mistral70B_Merge.sh",
    "ct-reports-for-1000-cases-ct-schema-mistral70b",
    "ALL_ct-reports-for-1000-cases-ct-schema-mistral70b.parquet",
    expected_shards=8,
)

print("PE schema vars:", len(pe_vars))
print("CT schema vars:", len(ct_vars))
print("Generated GPT, Mistral 7B, Mistral 70B, and merge scripts in:", OUTDIR)

PE schema vars: 18
CT schema vars: 4
Generated GPT, Mistral 7B, Mistral 70B, and merge scripts in: /nfs/turbo/umms-atjanke/liuwent/Notes_Feature_Abstraction/Notes_for_1000_cases
